# Operator Compiler Tutorial

Welcome to the **Operator Compiler Tutorial**!

This interactive notebook will guide you through the fundamentals of **performance engineering** using the XTC compiler, a research platform for optimizing AI operators.

By the end of this notebook, you will understand how to:

- Define computational graphs with XTC
- Compile and evaluate operator performance
- Explore the scheduling space to find optimal configurations

## Define your first graph

In XTC, computations are represented as **dataflow graphs**. A graph consists of:
- **Nodes** representing operations each implementing a specific computation (Operators)
- **Edges** representing data dependencies between operations (through Tensors)

Where:
- **Tensors** are multi-dimensional arrays that hold data
- **Operators** are tensor operations (e.g., matrix multiplication, convolution)

Let us start by creating a simple matrix multiplication graph. Matrix multiplication (matmul) computes $C = A \times B$ where:
- $A$ is an $I \times K$ matrix
- $B$ is a $K \times J$ matrix
- $C$ is the resulting $I \times J$ matrix

**Practice.** The code below is editable and the output (i.e. the serialized graph) is dynamically computed.

1. Try modifying the dimensions or data type (float32, float64) to see how the graph changes!
2. Add a ReLU activation after the matrix multiplication. Hint: `O.matmul()` returns a tensor that can be passed to another operator. Using ReLu: `R = O.relu(inp, name)`.

In [1]:
import xtc.graphs.xtc.op as O
from xtc.graphs.xtc.graph import XTCGraph

def matmul_graph(I: int, J: int, K: int, dtype: str) -> XTCGraph:
   """Create a graph computing C = A @ B."""
   a = O.tensor((I, K), dtype, name="A")
   b = O.tensor((K, J), dtype, name="B")
   with O.graph(name="matmul") as gb:
     C = O.matmul(a, b, name="C")
   return gb.graph

I, J, K, dtype = 16, 1000, 512, "float32"
graph = matmul_graph(I=I,J=J,K=K,dtype=dtype)

print(graph)

graph:
  name: matmul
  inputs:
  - %0 : 16x512xfloat32
  - %1 : 512x1000xfloat32
  outputs:
  - %2 : 16x1000xfloat32
  nodes:
  - %2: matmul(%0, %1) {name = 'C'} : [16x512xfloat32, 512x1000xfloat32] -> [16x1000xfloat32]



## Compile and Evaluate

Now that we have a graph, we can compile it and measure its baseline performance (without any optimization).

The compilation pipeline in XTC follows these steps:
1. **Create a Backend**: In XTC, the backend corresponds to an existing framework such as MLIR or TVM that, given a schedule, can generate the code for a specific target
2. **Get a Scheduler**: In XTC, a scheduler is a builder that creates a schedule. Even without optimizations, we need a scheduler to get a default loop structure.
3. **Compile**: Generate executable code
4. **Evaluate**: Run the compiled code and measure performance

**Practice.** The script below compiles the matmul graph without any optimization and displays both the generated code (if you unroll the accordion) and its performance on chip.

1. Use the radio buttons to select optionally an output (IR, generated assembly or equivalent optimized C code)
2. *Inspect the generated code.* Look at the Source IR, Transformed IR, Assembly and C. Are you able to identify the different loops i, j, k ?
3. *Observe the performance.* In your opinion, why is it so poor?

**Note:** Performance is measured as a percentage of peak (the estimated FLOP/s of one CPU core). The latter is by automatically approximated by running a benchmark, but this technique comes with (little) noise.

In [13]:
import ipy
ipy.display_nradio("print", "Print options for the execution of next cell", "none", "source IR", "transformed IR", "Assembly", "C")

RadioButtons(description='Print options for the execution of next cell', options=(('none', 'none'), ('source I…

In [12]:
import ipy
import xtc.graphs.xtc.op as O
from xtc.graphs.xtc.graph import XTCGraph
from xtc.backends.tvm import Backend
from xtc.runtimes.host import HostRuntime
from pathlib import Path

def matmul_graph(I: int, J: int, K: int, dtype: str) -> XTCGraph:
   """Create a graph computing C = A @ B."""
   a = O.tensor((I, K), dtype, name="A")
   b = O.tensor((K, J), dtype, name="B")
   with O.graph(name="matmul") as gb:
     C = O.matmul(a, b, name="C")
   return gb.graph

# Problem setup
I, J, K, dtype = 16, 1000, 512, "float32"
base = "matmul_naive"

graph = matmul_graph(I, J, K, dtype)

# Compile (no transformations, just default loop structure)
backend = Backend(graph)
scheduler = backend.get_scheduler()
schedule = scheduler.schedule()

print_opts = dict(
    print_source_ir=ipy.nradio_value("print", "source IR"),
    print_transformed_ir=ipy.nradio_value("print", "transformed IR"),
    print_assembly=ipy.nradio_value("print", "Assembly"),
    emit_c=ipy.nradio_value("print", "C"),
)
compiler = backend.get_compiler(dump_file=base, shared_lib=True, **print_opts)
module = compiler.compile(schedule)

if ipy.nradio_value("print", "C"):
    print(Path(f"{base}.c").read_text())

# Evaluate and display results
peak_flops = HostRuntime.get().evaluate_flops(dtype)
evaluator = module.get_evaluator()
results, _, _ = evaluator.evaluate()
perf = (I * J * K) / min(results) / peak_flops * 100

print(f"\nPeak perfromance: {perf:.2f} %")

O = obj['C']
i, j, = O.op.axis
k, = O.op.reduce_axis
sch[O].reorder(i, j, k)

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def main(_24_: T.Buffer((16, 512), "float32"), _25_: T.Buffer((512, 1000), "float32"), C: T.Buffer((16, 1000), "float32")):
        T.func_attr({"from_legacy_te_schedule": T.bool(True), "tir.noalias": T.bool(True)})
        for i, j in T.grid(16, 1000):
            C_1 = T.Buffer((16000,), data=C.data)
            C_1[i * 1000 + j] = T.float32(0.0)
            for k in range(512):
                cse_var_1: T.int32 = i * 1000 + j
                _24__1 = T.Buffer((8192,), data=_24_.data)
                _25__1 = T.Buffer((512000,), data=_25_.data)
                C_1[cse_var_1] = C_1[cse_var_1] + _24__1[i * 512 + k] * _25__1[k * 1000 + j]

Peak perfromance: 1.76 %


## Optimize with Scheduling

The baseline compilation produces correct but unoptimized code. To improve performance, XTC exposes a **scheduler** with imperative primitives that transform the loop nest:

- `sch.interchange(["i", "k", "j"])` reorders the loops. Interchange improves memory access patterns by ensuring stride-1 access (contiguous memory) rather than strided access, maximizing cache efficiency. Along with primitive `sch.strip_mine` (see below) this allows to actually tile the loop body.
- `sch.strip_mine("j", {"j1": 16})` breaks loop `j` into smaller chunks of size 16. This transformation can also be seen as 1D tiling.
- To perform actual multi-dimensional tiling, combine `sch.strip_mine` with `sch.interchange`. For example, 2D tiling:
  ```python
  sch.strip_mine("j", {"j1": 16})
  sch.strip_mine("k", {"k1": 16})
  sch.interchange(["i", "j", "k", "j1", "k1"])
  ```
  This creates $16 \times 16$ tiles over the $(j, k)$ dimensions.
- `sch.vectorize(["j1"])` vectorizes the computation along the loop `j1`. Vectorization uses SIMD instructions to process multiple elements in parallel, significantly increasing throughput on modern CPUs.
- `sch.unroll({"j1":1})` unrolls the loop `j1` with an unroll factor of 1 (which has no effect). Unrolling reduces loop overhead (fewer branches) and exposes more instruction-level parallelism to the hardware.
- `sch.parallelize(["i"])` execute in parallel along the loop `i`. Parallelization is only useful on a multiple cores architecture and actually splits and dispatch the work onto the available cores.
- `sch.split("j", {"j0": 0, "j1": 16})` splits `j` into two segments, creating `j0` (iterations 0-15) and `j1` (iterations 16+). Splitting is useful for applying different transformations to different parts of a loop (e.g., vectorize the main part, handle the remainder separately).

**Practice.** The code below lets you define a schedule using imperative primitives. Use the radio buttons to select the backend and what IR to display. Compare the performance and generated code with the unoptimized version from the previous section.

1. *Transform the code.* Start with simple transformations and build up.
2. *Inspect the generated code.* How does the Assembly or IR differ from the baseline?
3. *Try to maximize the performance!*

*Hint: You may want to create a register tile (i1,j1), j1 being a multiple of your SIMD registers size. Generally 8.*

In [14]:
import ipy
ipy.display_nradio("print", "Print options for the execution of next cell", "none", "source IR", "transformed IR", "Assembly", "C")

RadioButtons(description='Print options for the execution of next cell', options=(('none', 'none'), ('source I…

In [9]:
import ipy
import xtc.graphs.xtc.op as O
from xtc.graphs.xtc.graph import XTCGraph
from xtc.backends.tvm import Backend
from xtc.runtimes.host import HostRuntime
from pathlib import Path

def matmul_graph(I: int, J: int, K: int, dtype: str) -> XTCGraph:
   """Create a graph computing C = A @ B."""
   a = O.tensor((I, K), dtype, name="A")
   b = O.tensor((K, J), dtype, name="B")
   with O.graph(name="matmul") as gb:
     C = O.matmul(a, b, name="C")
   return gb.graph

# Schedule definition
def schedule(sch):
    """Apply transformations to the scheduler."""
    sch.set_dims(['i','j','k'])
    # Add transformations here, e.g.:
    sch.strip_mine("j", {"j1": 4})
    sch.vectorize(["j1"])
    sch.interchange(["i", "k", "j", "j1"])

# Problem setup
I, J, K, dtype = 16, 1000, 512, "float32"
base = "matmul_opt"

graph = matmul_graph(I, J, K, dtype)

# Compile
backend = Backend(graph)
scheduler = backend.get_scheduler()
schedule(scheduler)
sched = scheduler.schedule()

print_opts = dict(
    print_source_ir=ipy.nradio_value("print", "source IR"),
    print_transformed_ir=ipy.nradio_value("print", "transformed IR"),
    print_assembly=ipy.nradio_value("print", "Assembly"),
    emit_c=ipy.nradio_value("print", "C"),
)
compiler = backend.get_compiler(dump_file=base, shared_lib=True, **print_opts)
module = compiler.compile(sched)

if ipy.nradio_value("print", "C"):
    print(Path(f"{base}.c").read_text())

# Evaluate and display results
peak_flops = HostRuntime.get().evaluate_flops(dtype)
evaluator = module.get_evaluator()
results, _, _ = evaluator.evaluate()
perf = (I * J * K) / min(results) / peak_flops * 100

print(f"\nPeak perfromance: {perf:.2f} %")

Disassembly of section .text:

<matmul_compute_>:
	push   %r15
	push   %r14
	push   %r13
	push   %r12
	push   %rbx
	mov    %rdx,%rbx
	mov    %rsi,%r14
	mov    %rdi,%r15
	xor    %r12d,%r12d
	mov    %rdi,%r13
	nopl   0x0(%rax,%rax,1)
	imul   $0xfa0,%r12,%rdi
	add    %r15,%rdi
	mov    $0xfa0,%edx
	xor    %esi,%esi
	call   <memset@plt>
	mov    %r12,%rax
	shl    $0xb,%rax
	add    %r14,%rax
	mov    %rbx,%rcx
	xor    %edx,%edx
	data16 cs nopw 0x0(%rax,%rax,1)
	vbroadcastss (%rax,%rdx,4),%xmm0
	mov    $0x40,%esi
	nopl   0x0(%rax,%rax,1)
	vmovaps -0x40(%rcx,%rsi,1),%xmm1
	vfmadd213ps -0x40(%r13,%rsi,1),%xmm0,%xmm1
	vmovaps %xmm1,-0x40(%r13,%rsi,1)
	vmovaps -0x30(%rcx,%rsi,1),%xmm1
	vfmadd213ps -0x30(%r13,%rsi,1),%xmm0,%xmm1
	vmovaps %xmm1,-0x30(%r13,%rsi,1)
	vmovaps -0x20(%rcx,%rsi,1),%xmm1
	vfmadd213ps -0x20(%r13,%rsi,1),%xmm0,%xmm1
	vmovaps %xmm1,-0x20(%r13,%rsi,1)
	vmovaps -0x10(%rcx,%rsi,1),%xmm1
	vfmadd213ps -0x10(%r13,%rsi,1),%xmm0,%xmm1
	vmovaps %xmm1,-0x10(%r13,%rsi,1)
	vmovaps (%rcx,%r